# Regional TTS ViTs Train

## Miniconda

In [1]:
%%bash
set -e
MINICONDA_PREFIX=/opt/conda

if [ -x "${MINICONDA_PREFIX}/bin/conda" ]; then
  echo "Miniconda already present at ${MINICONDA_PREFIX}"
else
  MINICONDA_INSTALLER_SCRIPT=Miniconda3-latest-Linux-x86_64.sh
  wget https://repo.anaconda.com/miniconda/${MINICONDA_INSTALLER_SCRIPT} -q
  bash ${MINICONDA_INSTALLER_SCRIPT} -b -p ${MINICONDA_PREFIX}
  rm -f ${MINICONDA_INSTALLER_SCRIPT}
fi

${MINICONDA_PREFIX}/bin/conda --version

PREFIX=/opt/conda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda
conda 26.1.1


## Env Python 3.11 and dependencies

In [2]:
import os
import shutil
import subprocess
from pathlib import Path

CONDA_BIN  = Path("/opt/conda/bin/conda")
ENV_NAME   = "regional_tts"
ENV_PREFIX = Path(f"/opt/conda/envs/{ENV_NAME}")
ENV_PYTHON = ENV_PREFIX / "bin" / "python"

os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]

def run(cmd):
    print("+ " + " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True)

# Accept conda ToS
run([CONDA_BIN, "tos", "accept", "--override-channels", "--channel", "https://repo.anaconda.com/pkgs/main"])
run([CONDA_BIN, "tos", "accept", "--override-channels", "--channel", "https://repo.anaconda.com/pkgs/r"])

# Create environment
if not ENV_PREFIX.exists():
    run([CONDA_BIN, "create", "-y", "-n", ENV_NAME, "python=3.11"])
else:
    print(f"Conda env already exists: {ENV_PREFIX}")

+ /opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
+ /opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
+ /opt/conda/bin/conda create -y -n regional_tts python=3.11
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/conda/envs/regional_tts

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.3.19  |       h06a4308_0         126 KB
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libexpat-2.7.5             |       h7354ed3_0         122 KB
    li



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda






python-3.11.15       | 29.5 MB   |            |   1% 
openssl-3.5.6        | 5.6 MB    | 3          |   3% 


sqlite-3.51.2        | 1.2 MB    | ####9      |  49% 



pip-26.0.1           | 1.1 MB    | #######5   |  75% 



pip-26.0.1           | 1.1 MB    | ########## | 100% 




ld_impl_linux-64-2.4 | 725 KB    | 2          |   2% 


sqlite-3.51.2        | 1.2 MB    | ########## | 100% 




ld_impl_linux-64-2.4 | 725 KB    | ########## | 100% 





packaging-26.0       | 197 KB    | 8          |   8% 






ca-certificates-2026 | 126 KB    | #2         |  13% 





packaging-26.0       | 197 KB    | ########## | 100% 






python-3.11.15       | 29.5 MB   | ##         |  20% 







libexpat-2.7.5       | 122 KB    | #3         |  13% 







libexpat-2.7.5       | 122 KB    | ########## | 100% 


sqlite-3.51.2        | 1.2 MB    | ########## | 100% 


sqlite-3.51.2        | 1.2 MB    | ########## | 100% 








tzdata-2026a         | 117 KB    | #3         |  14% 








tzdata

In [3]:
# Copy TTS code to working directory
src = Path("/kaggle/input/datasets/phanvin/regional-tts-code")
dst = Path("/kaggle/working/TTS")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print("Working dir:", Path.cwd())

Working dir: /kaggle/working/TTS


In [4]:
# Install dependencies
run([ENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
run([ENV_PYTHON, "-m", "pip", "install", "torch", "torchaudio"])
run([ENV_PYTHON, "-m", "pip", "install", "-e", "."])
run([ENV_PYTHON, "-m", "pip", "install", "numpy", "tensorboard", "jiwer", "pymcd", "librosa", "pyworld", "pymcd"])

# Verify
run([ENV_PYTHON, "-c", "import torch; print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"])
print("\nSetup complete. Python:", ENV_PYTHON)

+ /opt/conda/envs/regional_tts/bin/python -m pip install --upgrade pip setuptools wheel
+ /opt/conda/envs/regional_tts/bin/python -m pip install torch torchaudio
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 43.4 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 41.3 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 63.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 62.4 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 72.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 67.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 90.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 79.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 80.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.0 MB/s  0:

In [5]:
%%bash
export MPLBACKEND=agg
/opt/conda/envs/regional_tts/bin/python -c '
import os
from TTS.utils.manage import ModelManager
os.environ["TTS_HOME"] = "/kaggle/working/base_models"
manager = ModelManager()
model_path, _, _ = manager.download_model("tts_models/en/vctk/vits")
print("\n" + "="*50)
print(f"Model path: {model_path}")
print("="*50)
'


 > Downloading model to /kaggle/working/base_models/tts/tts_models--en--vctk--vits
 > Model's license - apache 2.0
 > Check https://choosealicense.com/licenses/apache-2.0/ for more info.

Model path: /kaggle/working/base_models/tts/tts_models--en--vctk--vits/model_file.pth


/opt/conda/envs/regional_tts/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Training

In [22]:
import os
import re
import subprocess
import threading
from pathlib import Path
from IPython.display import display
from tqdm.notebook import tqdm

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TQDM_DISABLE"]     = "0"

# ===================== HYPERPARAMETERS =====================
RUN_NAME        = "vits_vi_regional_pitch"
BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 16
EPOCHS          = 100
LR              = 5e-5
PITCH_ALPHA     = 0.1
COMPUTE_F0      = True
# /kaggle/working/base_models/tts/tts_models--en--vctk--vits/model_file.pth
RESTORE_PATH    = ""
CONTINUE_PATH   = "/kaggle/working/tts_train_output/vits_vi_regional_pitch-April-20-2026_03+40AM-dbf1a08a"
# ============================================================

PYTHON     = "/opt/conda/envs/regional_tts/bin/python"
CONDA_BIN  = "/opt/conda/bin/conda"
ENV_NAME   = "regional_tts"
CODE_DIR   = Path("/kaggle/working/TTS")
DATA_ROOT  = Path("/kaggle/input/datasets/nhtlnguyn1106/xttsv2-finetuning-data-20260417")
OUTPUT_DIR = Path("/kaggle/working/tts_train_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLBACKEND"] = "Agg"

# Detect GPUs
gpu_count = int(subprocess.check_output(
    [PYTHON, "-c", "import torch; print(torch.cuda.device_count())"], text=True
).strip())
assert gpu_count > 0, "No GPU! Enable GPU in Kaggle settings."
print(f"GPUs: {gpu_count}")

# ---- Minimal compatibility fixes (idempotent) ----
vits_path  = CODE_DIR / "TTS/tts/models/vits.py"
train_path = CODE_DIR / "recipes/vietnamese/vits/train_vits_vi.py"

for p in [vits_path, train_path, DATA_ROOT]:
    assert p.exists(), f"Missing: {p}"

vits_code  = vits_path.read_text(encoding="utf-8")
train_code = train_path.read_text(encoding="utf-8")

if "x = x[0, :].unsqueeze(0)" not in vits_code:
    vits_code = vits_code.replace(
        "x, sr = torchaudio.load(file_path)",
        "x, sr = torchaudio.load(file_path)\n    if x.shape[0] > 1:\n        x = x[0, :].unsqueeze(0)",
    )
    vits_path.write_text(vits_code, encoding="utf-8")
    print("[fix] mono audio in load_audio()")

if "PITCH_FMIN = 1.0" in train_code:
    train_code = train_code.replace("PITCH_FMIN = 1.0", "PITCH_FMIN = 65.0")
    train_path.write_text(train_code, encoding="utf-8")
    print("[fix] PITCH_FMIN 1.0 -> 65.0")

if gpu_count > 1 and "parse_known_args" not in train_code:
    train_code = train_code.replace(
        "return parser.parse_args()", "return parser.parse_known_args()[0]"
    )
    train_path.write_text(train_code, encoding="utf-8")
    print("[fix] parse_args -> parse_known_args for DDP")

# ---- Build training command ----
train_args = [
    "--dataset-path",    str(DATA_ROOT),
    "--metadata-train",  "train_wav.csv",
    "--metadata-val",    "eval.csv",
    "--run-name",        RUN_NAME,
    "--batch-size",      str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--epochs",          str(EPOCHS),
    "--lr",              str(LR),
    "--output-path",     str(OUTPUT_DIR),
    "--num-loader-workers", str(26),
    "--num-eval-loader-workers", str(13),
    "--patience", str(20)
    
]
if COMPUTE_F0:
    train_args += ["--compute-f0", "--f0-cache-path", str(OUTPUT_DIR / "f0_cache")]
if PITCH_ALPHA > 0:
    train_args += ["--pitch-loss-alpha", str(PITCH_ALPHA)]
if CONTINUE_PATH:
    train_args += ["--continue-path", CONTINUE_PATH]
if RESTORE_PATH:
    train_args += ["--restore-path", RESTORE_PATH]
# if RESTORE_VIETNAMESE_PATH:
#     train_args += ["--restore-vietnamese-path", RESTORE_VIETNAMESE_PATH]
os.chdir(CODE_DIR)

if gpu_count > 1:
    gpus = ",".join(str(i) for i in range(gpu_count))
    cmd = [CONDA_BIN, "run", "--no-capture-output", "-n", ENV_NAME,
           "python", "-m", "trainer.distribute", "--gpus", gpus,
           "--script", str(train_path)] + train_args
else:
    cmd = [PYTHON, str(train_path)] + train_args

if CONTINUE_PATH:
    status = "RESUME"
elif RESTORE_PATH:
    status = "FINE-TUNE"
else:
    status = "FROM SCRATCH"
print(f"\n{'='*60}")
print(f"{status} | gpus={gpu_count} | pitch_alpha={PITCH_ALPHA} | epochs={EPOCHS}")
print(f"{'='*60}\n")

# ─── Regex patterns ────────────────────────────────────────────────────────────
RE_F0 = re.compile(
    r"(\d+)/(\d+)\s+\[[\d:]+<[\d:]+,\s*([\d.]+)([a-z/]+)\]"
)
# Training step:  "   >> STEP: 150  - loss_0: 12.34  ..."  or " > STEP: 150"
RE_STEP = re.compile(r"STEP:\s*(\d+)")
# Training epoch: " > EPOCH: 3/600"
RE_EPOCH = re.compile(r"EPOCH:\s*(\d+)/(\d+)")
# Loss summary:   "   >> loss_gen: 8.23  loss_disc: 2.11  ..."
RE_LOSS  = re.compile(r"loss_\w+:\s*[\d.]+")

# ─── Progress bars (created lazily) ───────────────────────────────────────────
pb_f0    = None   # F0 precompute
pb_epoch = None   # training epochs
pb_step  = None   # steps within current epoch

last_step  = [0]
total_step = [None]   # filled once we know steps_per_epoch

def parse_and_display(line: str):
    global pb_f0, pb_epoch, pb_step

    # Strip carriage returns — the root cause of the messy output
    line = line.replace("\r", "").strip()
    if not line:
        return

    # ── F0 precompute progress ──────────────────────────────────────────────
    m = RE_F0.search(line)
    if m and pb_epoch is None:   # only before training starts
        cur, total, speed, unit = int(m.group(1)), int(m.group(2)), m.group(3), m.group(4)
        if pb_f0 is None:
            pb_f0 = tqdm(total=total, desc="  F0 precompute",
                         unit="file", dynamic_ncols=True,
                         bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
        delta = cur - pb_f0.n
        if delta > 0:
            pb_f0.update(delta)
        pb_f0.set_postfix(speed=f"{speed}{unit}")
        return

    # ── Epoch progress ──────────────────────────────────────────────────────
    m_ep = RE_EPOCH.search(line)
    if m_ep:
        ep_cur, ep_total = int(m_ep.group(1)), int(m_ep.group(2))
        if pb_f0 and not pb_f0.disable:
            pb_f0.close()   # F0 done, close that bar
        if pb_epoch is None:
            pb_epoch = tqdm(total=ep_total, desc="  Epoch",
                            unit="epoch", dynamic_ncols=True)
        delta = ep_cur - pb_epoch.n
        if delta > 0:
            pb_epoch.update(delta)
        # Reset step bar each epoch
        if pb_step is not None:
            pb_step.reset()
        return

    # ── Step progress ───────────────────────────────────────────────────────
    m_st = RE_STEP.search(line)
    if m_st and pb_epoch is not None:
        step = int(m_st.group(1))
        if pb_step is None:
            # Estimate steps_per_epoch from first step hit
            pb_step = tqdm(desc="    Step", unit="step",
                           dynamic_ncols=True)
        delta = step - last_step[0]
        if delta > 0:
            pb_step.update(delta)
        last_step[0] = step

        # Attach any loss info found on this line
        losses = RE_LOSS.findall(line)
        if losses:
            pb_step.set_postfix_str("  ".join(losses[:4]))  # show up to 4 losses
        return

    # ── All other lines: print as-is (setup logs, errors, summaries) ────────
    # Skip pure tqdm carriage-return lines that slipped through
    if line.startswith("%|") or line.startswith("| "):
        return
    print(line)

# ─── Run subprocess and parse output ──────────────────────────────────────────
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
)

try:
    for raw_line in proc.stdout:
        # Each raw_line may contain multiple \r-separated segments
        for segment in raw_line.split("\r"):
            parse_and_display(segment)
finally:
    ret = proc.wait()
    # Close all bars cleanly
    for pb in [pb_step, pb_epoch, pb_f0]:
        if pb is not None:
            pb.close()

if ret != 0:
    raise RuntimeError(f"Training failed (exit code {ret})")

print("\n Training complete")

GPUs: 1
[fix] mono audio in load_audio()

FINE-TUNE | gpus=1 | pitch_alpha=0.1 | epochs=100

/opt/conda/envs/regional_tts/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
import pkg_resources
Lazy F0 mode enabled via TTS_LAZY_F0=1
> Setting up Audio Processor...
> initialization of speaker-embedding layers.
Loading pretrained checkpoint (Transfer Learning): /kaggle/working/tts_train_output/vits_vi_regional_pitch-April-20-2026_03+40AM-dbf1a08a/best_model.pth
Loaded pretrained tensors: 978
Loaded speaker-related tensors: 1
Skipped speaker_encoder tensors: 0
Skipped tensors: 1
Missing keys after load: 1
Unexpected keys after load: 0
> Training Environment:
> Start Tensorboard: tensorboard --logdir=/kaggle/working/tts_train_output/vits_vi_regional_pit

  Epoch:   0%|          | 0/100 [00:00<?, ?epoch/s]

--> /kaggle/working/tts_train_output/vits_vi_regional_pitch-April-20-2026_09+48AM-dbf1a08a
[*] Lazy F0 mode enabled. Skipping full F0 precompute.
[!] pitch_stats.npy not found. Continuing with unnormalized F0.
> DataLoader initialization
|
> Using weighted sampler for attribute 'speaker_name' with alpha '1.0'
None
> Attribute weights for '['@HUE', '@QuangDien', '@speaker']'
/opt/conda/envs/regional_tts/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 26 worker processes in total. Our suggested max number of worker in current system is 24, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
self.check_worker_number_rationality()
 > TRAINING (2026-04-20 09:49:11) 
/opt/conda/envs/regional_tts/lib/python3.11/site-packages/torch/utils/data/dataloader.py:43

    Step: 0step [00:00, ?step/s]

- Tỷ lệ trẻ em được học vỡ lòng, tại nhà trường: 2,7%
[!] Character '%' not found in the vocabulary. Discarding it.
Trong tinh thần đó, Nhà xuất bản Trẻ xin trân trọng giới thiệu cùng bạn đọc xa gần tập sách: Đồng Bằng Sông Cửu Long - Nét Sinh Hoạt Xưa & Văn Minh Miệt Vườn.
[!] Character '&' not found in the vocabulary. Discarding it.
đồ hộ tr >> vâng và vừa lúc nãy thì Huế R cũng đã
[!] Character '>' not found in the vocabulary. Discarding it.
233.987 người so với 35.000 người hồi 15 năm trước tăng gần 700%. Năm
[!] Character '%' not found in the vocabulary. Discarding it.
phá lớn của Việt Nam và nó chiếm khoảng 11% diện tích đầm phá à ven bờ của cả
[!] Character '%' not found in the vocabulary. Discarding it.
233.987 người so với 35.000 người hồi 15 năm trước tăng gần 700%. Năm
[!] Character '%' not found in the vocabulary. Discarding it.
>> [âm nhạc] >> Tuy nhiên thì hôm nay Huế Review không
[!] Character '>' not found in the vocabulary. Discarding it.
Trong tinh thần đó, Nhà xuất b

RuntimeError: Training failed (exit code 1)

In [7]:
import csv
import os
import pathlib
import subprocess

ENV_PYTHON  = "/opt/conda/envs/regional_tts/bin/python"
DATA_ROOT   = pathlib.Path("/kaggle/input/datasets/nhtlnguyn1106/xttsv2-finetuning-data-20260417")
OUTPUT_DIR = pathlib.Path("/kaggle/working/")
CODE_DIR = pathlib.Path("/kaggle/input/datasets/phanvin/regional-tts-code")
MAX_SAMPLES = 0

model_path  = pathlib.Path("/kaggle/input/datasets/phanvin/regional-tts-model/best_model.pth")
config_path = pathlib.Path("/kaggle/input/datasets/phanvin/regional-tts-model/config.json")
assert model_path.exists(), f"Missing {model_path}"
assert config_path.exists(), f"Missing {config_path}"

def build_pairs_csv(meta_path, data_root, out_path):
    print(f"Building {out_path.name}...")
    pairs = []
    with open(meta_path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="|")
        fields = set(reader.fieldnames or [])
        audio_key = "audio_file" if "audio_file" in fields else "wav_file"
        for row in reader:
            audio = (row.get(audio_key) or "").strip()
            text = (row.get("text") or "").strip()
            spk = (row.get("speaker_name") or "").strip()
            if not audio or not text:
                continue
            if not audio.startswith("/"):
                audio_path = data_root / audio
            else:
                audio_path = pathlib.Path(audio)
            pairs.append({"text": text, "ref_wav": str(audio_path), "speaker_name": spk})
            
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["text", "ref_wav", "speaker_name"], delimiter="|")
        writer.writeheader()
        writer.writerows(pairs)

build_pairs_csv(DATA_ROOT / "test_wav.csv", DATA_ROOT, OUTPUT_DIR / "pairs_test.csv")

pairs_csv = OUTPUT_DIR / "pairs_test.csv"
if not pairs_csv.exists():
    print(f"pairs_test.csv not found, cannot evaluate!")
else:
    eval_out = OUTPUT_DIR / "eval_test"
    print(f"\n{'='*40}\nEvaluating TEST set\n{'='*40}")
    
    cmd = [
        ENV_PYTHON,
        str(CODE_DIR / "recipes/vietnamese/vits/evaluate.py"),
        "--model-path",  str(model_path),
        "--config-path", str(config_path),
        "--pairs-csv",   str(pairs_csv),
        "--output-dir",  str(eval_out),
        "--gpu",
    ]
    if MAX_SAMPLES > 0:
        cmd += ["--max-samples", str(MAX_SAMPLES)]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        print(f"WARNING: TEST evaluation failed")

def read_summary(path):
    p = pathlib.Path(path)
    if not p.exists():
        return None
    with open(p, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        if row.get("speaker_name") == "ALL":
            return row
    return rows[0] if rows else None

test_s = read_summary(OUTPUT_DIR / "eval_test/acoustic_eval_summary.csv")

if test_s:
    print(f"\n{'='*45}")
    print(f"{'Acoustic Metric':<20}{'Test Set':>20}")
    print(f"{'-'*45}")
    print(f"{'Samples':<20}{test_s['num_samples']:>20}")
    print(f"{'MCD (dB)':<20}{float(test_s['mean_mcd_db']):>20.4f}")
    print(f"{'FFE Rate':<20}{float(test_s['mean_ffe_rate']):>20.4f}")
    print(f"{'DTW Cost':<20}{float(test_s['mean_dtw_cost']):>20.4f}")
    print(f"{'='*45}")
else:
    print("\nNo output summary found for the test set.")


Building pairs_test.csv...

Evaluating TEST set
/opt/conda/envs/regional_tts/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/kaggle/working/TTS/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:65